In [1]:
from pathlib import Path
print(Path.cwd())

C:\Users\karol\Bakalarka26\prakticka_cast\notebooks\01_time_series


In [2]:
from datetime import datetime    # import knižníc
from pathlib import Path
import platform, sys
import sys
sys.path.append(str(Path.cwd().parents[1]))  # pridá koreň projektu (prakticka_cast) na sys.path


DOMAIN = "ts"             # časové rady
DATASET_NAME = "noaa_ghcn_2020_2024"     # názov datasetu
LIBRARY = "matplotlib"        # knižnica
TASK_ID = "trend_a_sezonnost"   # úloha
N_RUNS = 10                   # počet vykreslení
SEED = 42                    # kvoli reprodukovateľnosti
TIMESTAMP = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")   # čas spustenia


In [3]:
import os, time, json, csv
import numpy as np, pandas as pd, statistics as stats
import matplotlib, matplotlib.pyplot as plt, seaborn as sns
import plotly
import plotly.express as px, plotly.graph_objects as go


try:
    import geopandas as gpd   # ak nie je geopandas, = none
except Exception:
    gpd = None
try:
    import yaml
except Exception:
    yaml = None

plt.rcParams.update({           # nastavenie globálnych štýlov pre matpplotlib
    "figure.dpi": 120, "figure.figsize": (8, 4.5),
    "axes.titlesize": 12, "axes.labelsize": 10,
    "legend.fontsize": 9, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "font.family": "DejaVu Sans"
})
sns.set_theme(style="whitegrid")   # štýl pre seaborn
px.defaults.template = "plotly_white"   # štýly pre plotly
px.defaults.width = 800
px.defaults.height = 450

In [4]:
from pathlib import Path
import plotly.io as pio

PROJECT_ROOT = Path.cwd().parents[1]
OUT_ROOT = PROJECT_ROOT / "outputs"

FIG_DIR = OUT_ROOT / "figures"
HTML_DIR = OUT_ROOT / "html"
METRICS_DIR = OUT_ROOT / "metrics"
TABLES_DIR = OUT_ROOT / "tables"

CFG_PLOTS = {"dpi": 120, "svg": True}

for d in [FIG_DIR, HTML_DIR, METRICS_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DOMAIN_MAP = {"ts": "time_series", "corr": "correlations", "geo": "geo"}
DOMAIN_DIR = DOMAIN_MAP.get(DOMAIN, DOMAIN)
LIB_DIR = LIBRARY.replace("_", "")

FIG_SUBDIR = FIG_DIR / DOMAIN_DIR / LIB_DIR
HTML_SUBDIR = HTML_DIR / DOMAIN_DIR

FIG_SUBDIR.mkdir(parents=True, exist_ok=True)
HTML_SUBDIR.mkdir(parents=True, exist_ok=True)

# pre Plotly geo export PNG/SVG
PLOTLY_TOPOJSON_DIR = PROJECT_ROOT / "assets" / "plotly_topojson"
if PLOTLY_TOPOJSON_DIR.exists():
    pio.defaults.topojson = str(PLOTLY_TOPOJSON_DIR.resolve())


def file_basename(prefix: str, variant: str, ext: str) -> Path:
    name = f"{DOMAIN}_{LIBRARY}_{DATASET_NAME}_{TASK_ID}_{variant}_{TIMESTAMP}.{ext}"
    return (HTML_SUBDIR if ext.lower() == "html" else FIG_SUBDIR) / name

class Timer:
    def __enter__(self):
        self.t0 = time.perf_counter()
        return self

    def __exit__(self, *args):
        self.elapsed = (time.perf_counter() - self.t0) * 1000.0

def file_size_kb(path: Path):
    try:
        return os.path.getsize(path) / 1024.0
    except OSError:
        return None

def write_metrics(row: dict, csv_path: Path = METRICS_DIR / "metrics.csv"):
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    exists = csv_path.exists()
    with open(csv_path, "a", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not exists:
            w.writeheader()
        w.writerow(row)

def count_loc_from_callable(fn) -> int:
    import inspect, textwrap
    try:
        src = textwrap.dedent(inspect.getsource(fn))
        lines = [ln for ln in src.splitlines() if ln.strip() and not ln.strip().startswith("#")]
        return len(lines)
    except Exception:
        return None

def runtime_meta():
    return {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "matplotlib": matplotlib.__version__,
        "seaborn": sns.__version__,
        "plotly": plotly.__version__
    }

In [5]:

from src.data.ghcn import load_noaa_monthly_tavg, BY_YEAR_DIR # import helper funkcie a priečinok s ročnými súbormi

REQUESTED_YEARS = list(range(2020, 2025))               # požadované roky a stanice
STATION_FILTERS = ["HURBANOVO", "POPRAD"]

print("Notebook hľadá ročné NOAA súbory v priečinku:")
print(BY_YEAR_DIR)

def existing_years(years):          # ktoré roky sú dostupné
    ok = [y for y in years if (BY_YEAR_DIR / f"{y}.csv.gz").exists()]
    missing = [y for y in years if y not in ok]
    if missing:
        print("Chýbajú súbory pre roky:", missing)
    return ok

YEARS = existing_years(REQUESTED_YEARS)   # len dostupné roky

if not YEARS:   # ak nie sú, = koniec
    raise FileNotFoundError(
        f"Nenašiel sa žiadny NOAA GHCN súbor v priečinku: {BY_YEAR_DIR}"
    )

def load_data(domain: str, dataset_name: str):
    if domain != "ts":
        raise ValueError("Tento notebook je určený pre časové rady (DOMAIN='ts').")

    df = load_noaa_monthly_tavg(  # načítanie mesačných dát
        years=YEARS,
        station_name_filters=STATION_FILTERS
    ).copy()

    df = df.sort_values("date").reset_index(drop=True)  # zoradenie podľa dátumu
    return df

DF = load_data(DOMAIN, DATASET_NAME) # výsledný dataframe

print("Použité roky:", YEARS)
print("Počet mesiacov:", len(DF))
display(DF.head())
display(DF.tail())
display(DF.info())

Notebook hľadá ročné NOAA súbory v priečinku:
C:\Users\karol\Bakalarka26\prakticka_cast\data\raw\time_series\ghcn\by_year
Použité stanice:
           station          name
41025  LO000011934  POPRAD/TATRY
41026  LOE00105562     HURBANOVO
Použité roky: [2020, 2021, 2022, 2023, 2024]
Počet mesiacov: 60


,date,value
0,2020-01-01,-1.551613
1,2020-02-01,3.818965
2,2020-03-01,4.798387
3,2020-04-01,9.750000
4,2020-05-01,12.100000


,date,value
55,2024-08-01,21.670000
56,2024-09-01,15.938334
57,2024-10-01,9.933871
58,2024-11-01,2.565385
59,2024-12-01,0.486667


<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    60 non-null     datetime64[us]
 1   value   60 non-null     float32       
dtypes: datetime64[us](1), float32(1)
memory usage: 852.0 bytes


None

In [6]:
def plot_baseline(df):  # základná verzia grafu
    if LIBRARY == "matplotlib":  # ak matplotlib
        fig, ax = plt.subplots()
        if DOMAIN == "ts":    # ak domena ts, vykreslí sa graf
            ax.plot(df["date"], df["value"], lw=1.5, color="#1f77b4")
            ax.set_title("Mesačný vývoj priemernej teploty (2020–2024)"); ax.set_xlabel("Dátum"); ax.set_ylabel("Teplota [°C]")
            fig.autofmt_xdate()
        elif DOMAIN == "corr":
            ax.scatter(df["x"], df["y"], s=8, alpha=0.6); ax.set_title("Scatter x–y – baseline")
        elif DOMAIN == "geo":
            base = df.plot(figsize=(6,6), markersize=5, alpha=0.6); fig = base.get_figure()
        return fig, "mpl"

    if LIBRARY == "seaborn":
        fig, ax = plt.subplots()
        if DOMAIN == "ts":
            sns.lineplot(df, x="date", y="value", ax=ax); ax.set_title("Časový rad – baseline (Seaborn)"); fig.autofmt_xdate()
        elif DOMAIN == "corr":
            sns.scatterplot(df, x="x", y="y", s=15, alpha=0.6, ax=ax); ax.set_title("Scatter x–y – baseline (Seaborn)")
        return fig, "mpl"

    if LIBRARY == "plotly":
        if DOMAIN == "ts":
            fig = px.line(df, x="date", y="value", title="Časový rad – baseline (Plotly)")
        elif DOMAIN == "corr":
            fig = px.scatter(df, x="x", y="y", opacity=0.7, title="Scatter x–y – baseline (Plotly)")
        elif DOMAIN == "geo":
            fig = px.scatter_geo(df, lon="lon", lat="lat", color="cat", opacity=0.7, title="Bodová mapa – baseline (Plotly)")
        return fig, "plotly"

    raise ValueError(f"Nepodporovaná knižnica: {LIBRARY}")

In [7]:
def prepare_ts_final(df, ma_window: int = 12):
    out = df.copy()
    out = out.sort_values("date").reset_index(drop=True)
    out["trend"] = out["value"].rolling(ma_window, min_periods=1, center=True).mean()
    return out

In [8]:
def plot_final(df):
    if LIBRARY in ("matplotlib","seaborn"):
        fig, ax = plt.subplots()
        if DOMAIN == "ts":
            w = 12
            df2 = prepare_ts_final(df, ma_window=w)
            ax.plot(df2["date"], df2["value"], color="#1f77b4", lw=1.0, alpha=0.5, label="Hodnota")
            ax.plot(df2["date"], df2["trend"], color="#d62728", lw=2.0, label=f"Trend (MA{w})")
            ax.set_title("Mesačný vývoj priemernej teploty s trendom (MA12)")
            ax.set_xlabel("Dátum")
            ax.set_ylabel("Teplota [°C]")
            ax.legend()
            ax.grid(True, alpha=0.3)
            fig.autofmt_xdate()
        elif DOMAIN == "corr":
            sns.kdeplot(df, x="x", y="y", levels=6, fill=True, cmap="Blues", alpha=0.7, ax=ax)
            sns.scatterplot(df.sample(min(500, len(df)), random_state=SEED), x="x", y="y", s=8, alpha=0.4, color="#2ca02c", ax=ax)
            ax.set_title("Závislosť x–y s hustotnými obálkami"); ax.grid(True, alpha=0.3)
        elif DOMAIN == "geo":
            if gpd is None: raise RuntimeError("Geopandas nie je dostupný.")
            base = df.plot(figsize=(7,7), alpha=0.7, markersize=6, column="cat", legend=True, categorical=True)
            ax = base.axes; ax.set_title("Bodová mapa kategórií"); ax.set_axis_off(); fig = base.get_figure()
        return fig, "mpl"

    

    raise ValueError(f"Nepodporovaná knižnica: {LIBRARY}")

In [9]:
def render_and_time(make_fig_callable, df, variant: str, n_runs: int = 5) -> dict:
    times, out_files = [], []

    html_path = None
    png_path = None
    svg_path = None

    # warm-up beh, nezapočítava sa do metrík
    fig, kind = make_fig_callable(df)
    if kind == "mpl":
        plt.close(fig)

    for _ in range(n_runs):
        with Timer() as t:
            fig, kind = make_fig_callable(df)
        times.append(t.elapsed)

        if kind == "mpl":
            plt.close(fig)

    fig, kind = make_fig_callable(df)
    export_elapsed_ms = None

    if kind == "mpl":
        png_path = file_basename("plot", variant, "png")
        with Timer() as t:
            fig.savefig(png_path, dpi=CFG_PLOTS.get("dpi", 120), bbox_inches="tight")
        export_elapsed_ms = t.elapsed
        plt.close(fig)
        out_files.append(png_path)

        if CFG_PLOTS.get("svg", True):
            svg_path = png_path.with_suffix(".svg")
            fig, _ = make_fig_callable(df)
            fig.savefig(svg_path, bbox_inches="tight")
            plt.close(fig)
            out_files.append(svg_path)

    elif kind == "plotly":
        html_path = file_basename("plot", variant, "html")
        with Timer() as t:
            fig.write_html(html_path, include_plotlyjs="cdn", full_html=True)
        export_elapsed_ms = t.elapsed
        out_files.append(html_path)

        try:
            png_path = file_basename("plot", variant, "png")
            fig.write_image(png_path, scale=2)
            out_files.append(png_path)

            if CFG_PLOTS.get("svg", True):
                svg_path = file_basename("plot", variant, "svg")
                fig.write_image(svg_path)
                out_files.append(svg_path)
        except Exception as e:
            print("Upozornenie: PNG/SVG export (Plotly) preskočený:", e)

    return {
        "build_ms_median": stats.median(times),
        "build_ms_p95": np.percentile(times, 95),
        "export_ms": export_elapsed_ms,
        "saved": str(html_path or png_path) if (html_path or png_path) else None,
        "html_path": str(html_path) if html_path else None,
        "png_path": str(png_path) if png_path else None,
        "svg_path": str(svg_path) if svg_path else None,
        "html_size_kb": round(file_size_kb(html_path), 2) if html_path else None,
        "png_size_kb": round(file_size_kb(png_path), 2) if png_path else None,
        "svg_size_kb": round(file_size_kb(svg_path), 2) if svg_path else None,
        "all_files": [str(p) for p in out_files]
    }

loc_baseline = count_loc_from_callable(plot_baseline)
loc_final    = count_loc_from_callable(plot_final)
res_baseline = render_and_time(plot_baseline, DF, "baseline", n_runs=N_RUNS)
res_final    = render_and_time(plot_final,    DF, "final",    n_runs=N_RUNS)

meta = runtime_meta()
common = {
    "timestamp": TIMESTAMP, "domain": DOMAIN, "dataset": DATASET_NAME, "library": LIBRARY, "task_id": TASK_ID,
    "python": meta["python"], "platform": meta["platform"],
    "matplotlib_ver": meta["matplotlib"], "seaborn_ver": meta["seaborn"], "plotly_ver": meta["plotly"]
}

write_metrics({
    **common,
    "variant": "baseline",
    "build_ms_median": round(res_baseline["build_ms_median"], 2),
    "build_ms_p95": round(res_baseline["build_ms_p95"], 2),
    "export_ms": round(res_baseline["export_ms"] or 0.0, 2),
    "loc": loc_baseline,
    "saved": res_baseline["saved"],
    "html_path": res_baseline["html_path"],
    "png_path": res_baseline["png_path"],
    "svg_path": res_baseline["svg_path"],
    "html_size_kb": res_baseline["html_size_kb"],
    "png_size_kb": res_baseline["png_size_kb"],
    "svg_size_kb": res_baseline["svg_size_kb"]
})

write_metrics({
    **common,
    "variant": "final",
    "build_ms_median": round(res_final["build_ms_median"], 2),
    "build_ms_p95": round(res_final["build_ms_p95"], 2),
    "export_ms": round(res_final["export_ms"] or 0.0, 2),
    "loc": loc_final,
    "saved": res_final["saved"],
    "html_path": res_final["html_path"],
    "png_path": res_final["png_path"],
    "svg_path": res_final["svg_path"],
    "html_size_kb": res_final["html_size_kb"],
    "png_size_kb": res_final["png_size_kb"],
    "svg_size_kb": res_final["svg_size_kb"]
})

print("Hotovo. Uložené:")
print("  baseline:", res_baseline["saved"])
print("  final   :", res_final["saved"])
print("Metriky ->", METRICS_DIR / "metrics.csv")

Hotovo. Uložené:
  baseline: C:\Users\karol\Bakalarka26\prakticka_cast\outputs\figures\time_series\matplotlib\ts_matplotlib_noaa_ghcn_2020_2024_trend_a_sezonnost_baseline_2026-04-01_11-08-58.png
  final   : C:\Users\karol\Bakalarka26\prakticka_cast\outputs\figures\time_series\matplotlib\ts_matplotlib_noaa_ghcn_2020_2024_trend_a_sezonnost_final_2026-04-01_11-08-58.png
Metriky -> C:\Users\karol\Bakalarka26\prakticka_cast\outputs\metrics\metrics.csv


In [10]:
m = pd.read_csv(METRICS_DIR / "metrics.csv")

subset = m[
    (m["timestamp"] == TIMESTAMP) &
    (m["domain"] == DOMAIN) &
    (m["dataset"] == DATASET_NAME) &
    (m["library"] == LIBRARY) &
    (m["task_id"] == TASK_ID)
].copy()

display(subset[[
    "variant",
    "build_ms_median",
    "build_ms_p95",
    "export_ms",
    "loc",
    "html_size_kb",
    "png_size_kb",
    "svg_size_kb",
    "saved",
    "html_path",
    "png_path",
    "svg_path"
]])

print("Metrics exists AFTER:", (METRICS_DIR / "metrics.csv").exists())
print("Metrics path:", METRICS_DIR / "metrics.csv")

,variant,build_ms_median,build_ms_p95,export_ms,loc,html_size_kb,png_size_kb,svg_size_kb,saved,html_path,png_path,svg_path
10,baseline,12.59,17.18,170.16,28,NaN,54.49,29.83,C:\Users\karol\Bakalarka26\prakticka_cast\outp...,NaN,C:\Users\karol\Bakalarka26\prakticka_cast\outp...,C:\Users\karol\Bakalarka26\prakticka_cast\outp...
11,final,14.08,25.44,142.18,24,NaN,62.64,35.48,C:\Users\karol\Bakalarka26\prakticka_cast\outp...,NaN,C:\Users\karol\Bakalarka26\prakticka_cast\outp...,C:\Users\karol\Bakalarka26\prakticka_cast\outp...


Metrics exists AFTER: True
Metrics path: C:\Users\karol\Bakalarka26\prakticka_cast\outputs\metrics\metrics.csv
